# RFM Görselleştirme
Bu notebook `rfm_segments.csv` dosyasını okur ve rapor için tüm grafikleri üretir.

In [2]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os

rfm = pd.read_csv('rfm_segments.csv')

os.makedirs('figures', exist_ok=True)

plt.rcParams.update({
    'font.family'      : 'DejaVu Serif',
    'font.size'        : 11,
    'axes.titlesize'   : 13,
    'axes.titleweight' : 'bold',
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'figure.dpi'       : 150,
})

PALETTE = ['#2c3e50','#2980b9','#27ae60','#e67e22','#e74c3c',
           '#8e44ad','#16a085','#f39c12']

print(f'Veri yüklendi: {len(rfm):,} müşteri, {rfm["Segment"].nunique()} segment')

Veri yüklendi: 4,338 müşteri, 8 segment


In [10]:
# ŞEKİL 1 — Segment Müşteri Sayısı

seg_count = rfm['Segment'].value_counts().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(seg_count.index, seg_count.values,
               color=PALETTE[:len(seg_count)], edgecolor='white', linewidth=0.8)
for bar, val in zip(bars, seg_count.values):
    ax.text(val + 8, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=10)
ax.set_xlabel('Müşteri Sayısı')
ax.set_title('Şekil 1. Müşteri Segmentlerinin Dağılımı')
ax.set_xlim(0, seg_count.max() * 1.15)
plt.tight_layout()
plt.savefig('figures/fig1_segment_dagilimi.png', bbox_inches='tight')
plt.show()

/var/folders/84/_5dfw6l50pxgsy8_1qs0_b0w0000gn/T/ipykernel_8759/2634149081.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
# ŞEKİL 2 — Gelir Pasta Grafiği

seg_rev = rfm.groupby('Segment')['Monetary'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 6))
wedges, texts, autotexts = ax.pie(
    seg_rev.values,
    labels=seg_rev.index,
    autopct=lambda p: f'%{p:.1f}' if p > 2 else '',
    colors=PALETTE[:len(seg_rev)],
    startangle=140,
    wedgeprops={'edgecolor':'white','linewidth':1.5}
)
for t in autotexts:
    t.set_fontsize(9)
ax.set_title('Şekil 2. Segmentlerin Toplam Gelire Katkısı')
plt.tight_layout()
plt.savefig('figures/fig2_gelir_pasta.png', bbox_inches='tight')
plt.show()

/var/folders/84/_5dfw6l50pxgsy8_1qs0_b0w0000gn/T/ipykernel_8759/2672253119.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# ŞEKİL 3 — R, F, M Dağılım Histogramları

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, col, label, color in zip(
        axes,
        ['Recency','Frequency','Monetary'],
        ['Recency (Gün)','Frequency (Sipariş Sayısı)','Monetary (£)'],
        ['#2980b9','#27ae60','#e67e22']):
    ax.hist(rfm[col], bins=40, color=color, edgecolor='white', linewidth=0.5)
    ax.axvline(rfm[col].median(), color='black', linestyle='--',
               linewidth=1.2, label=f'Medyan: {rfm[col].median():.0f}')
    ax.set_xlabel(label)
    ax.set_ylabel('Müşteri Sayısı')
    ax.legend(fontsize=9)

fig.suptitle('Şekil 3. R, F, M Metriklerinin Dağılımı', fontweight='bold')
plt.tight_layout()
plt.savefig('figures/fig3_rfm_dagilim.png', bbox_inches='tight')
plt.show()

/var/folders/84/_5dfw6l50pxgsy8_1qs0_b0w0000gn/T/ipykernel_8759/1059377945.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# ŞEKİL 4 — Normalize RFM Ortalamaları (Grouped Bar)

seg_rfm = rfm.groupby('Segment')[['Recency','Frequency','Monetary']].mean().round(1)
seg_rfm_norm = seg_rfm.copy()
for col in seg_rfm_norm.columns:
    seg_rfm_norm[col] = (seg_rfm_norm[col] - seg_rfm_norm[col].min()) / \
                         (seg_rfm_norm[col].max() - seg_rfm_norm[col].min())

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(seg_rfm_norm))
w = 0.25
ax.bar(x - w, seg_rfm_norm['Recency'],  w, label='Recency (norm)',   color='#2980b9')
ax.bar(x,     seg_rfm_norm['Frequency'],w, label='Frequency (norm)', color='#27ae60')
ax.bar(x + w, seg_rfm_norm['Monetary'], w, label='Monetary (norm)',  color='#e67e22')
ax.set_xticks(x)
ax.set_xticklabels(seg_rfm_norm.index, rotation=25, ha='right')
ax.set_ylabel('Normalize Değer (0-1)')
ax.set_title('Şekil 4. Segmentlerin Normalize RFM Ortalamaları')
ax.legend()
plt.tight_layout()
plt.savefig('figures/fig4_segment_rfm_ortalama.png', bbox_inches='tight')
plt.show()

/var/folders/84/_5dfw6l50pxgsy8_1qs0_b0w0000gn/T/ipykernel_8759/987489460.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# ŞEKİL 5 — Recency vs Frequency Scatter

fig, ax = plt.subplots(figsize=(9, 6))
segments = rfm['Segment'].unique()
colors_map = dict(zip(segments, PALETTE[:len(segments)]))

for seg in segments:
    sub = rfm[rfm['Segment'] == seg]
    ax.scatter(sub['Recency'], sub['Frequency'],
               label=seg, alpha=0.6, s=25,
               color=colors_map[seg], edgecolors='none')

ax.set_xlabel('Recency (Gün)')
ax.set_ylabel('Frequency (Sipariş Sayısı)')
ax.set_title('Şekil 5. Recency ve Frequency İlişkisi (Segment Bazlı)')
ax.legend(fontsize=8, markerscale=1.5, bbox_to_anchor=(1.01,1), loc='upper left')
plt.tight_layout()
plt.savefig('figures/fig5_recency_frequency_scatter.png', bbox_inches='tight')
plt.show()

/var/folders/84/_5dfw6l50pxgsy8_1qs0_b0w0000gn/T/ipykernel_8759/2901241747.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# ŞEKİL 6 — RFM Heatmap

pivot = rfm.groupby(['R_Score','F_Score'])['Monetary'].mean().unstack()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.5, ax=ax,
            cbar_kws={'label':'Ort. Monetary (£)'})
ax.set_xlabel('F Skoru')
ax.set_ylabel('R Skoru')
ax.set_title('Şekil 6. R-F Skor Kombinasyonuna Göre Ortalama Harcama (£)')
plt.tight_layout()
plt.savefig('figures/fig6_rfm_heatmap.png', bbox_inches='tight')
plt.show()

/var/folders/84/_5dfw6l50pxgsy8_1qs0_b0w0000gn/T/ipykernel_8759/2164786223.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
# ŞEKİL 7 — Segment Büyüklük ve Gelir Karşılaştırması

seg_ord = rfm.groupby('Segment').agg(
    Sayi  = ('CustomerID','count'),
    Gelir = ('Monetary','sum')
).sort_values('Gelir', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Şekil 7. Segment Büyüklüğü ve Gelir Karşılaştırması', fontweight='bold')

axes[0].bar(range(len(seg_ord)), seg_ord['Sayi'],
            color=PALETTE[:len(seg_ord)], edgecolor='white')
axes[0].set_xticks(range(len(seg_ord)))
axes[0].set_xticklabels(seg_ord.index, rotation=30, ha='right', fontsize=9)
axes[0].set_ylabel('Müşteri Sayısı')
axes[0].set_title('Müşteri Sayısı')

axes[1].bar(range(len(seg_ord)), seg_ord['Gelir']/1e6,
            color=PALETTE[:len(seg_ord)], edgecolor='white')
axes[1].set_xticks(range(len(seg_ord)))
axes[1].set_xticklabels(seg_ord.index, rotation=30, ha='right', fontsize=9)
axes[1].set_ylabel('Toplam Gelir (Milyon £)')
axes[1].set_title('Toplam Gelir')

plt.tight_layout()
plt.savefig('figures/fig7_segment_buyukluk_gelir.png', bbox_inches='tight')
plt.show()

print('\n✅ Tüm grafikler figures/ klasörüne kaydedildi.')


✅ Tüm grafikler figures/ klasörüne kaydedildi.


/var/folders/84/_5dfw6l50pxgsy8_1qs0_b0w0000gn/T/ipykernel_8759/3653155788.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
